# SLIDE-0330 model prediction gallery

This notebook reads the same small random crops from SLIDE-0330 and visualizes predictions from the four self-trained checkpoints and the public `fluorescence_nuclei_and_cells` model.

The eleven input channels and their order are copied from the SLIDE-0330 watershed test notebook. Every model sees the same crop in the same channel order. Self-trained checkpoints are exported to native TorchScript and evaluated through `InstanSeg.eval_small_image`, so this gallery avoids the external medium-mode tiling/stitching confounder.

The default is `resolve_cell_and_nucleus=False`, which shows the independently predicted nuclear and cell heads. Set the flag to `True` to inspect native InstanSeg reconciliation. Our experimental nucleus-seeded watershed is not used here.

In [ ]:
import gc
import json
import os
import re
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
import torch
import zarr
from IPython.display import display
from skimage.segmentation import find_boundaries

SLIDE_ID = 'SLIDE-0330'
FULL_MERGE_OME = Path(
    '/data1/lowes/ratnayn/Data/CellDive_analysis_data/image_data/'
    'SLIDE-0330/outputs_v3/SLIDE-0330_full_merge.ome.tif'
)
TRAINING_ROOT = Path(os.environ.get(
    'INSTANSEG_TRAINING_ROOT', '/data1/lowes/ratnayn/Data/instanseg'
)).expanduser().resolve()
MODEL_ROOT = TRAINING_ROOT / 'models'
SOURCE_ROOT = Path(os.environ.get(
    'INSTANSEG_EVAL_SOURCE_ROOT',
    '/data1/lowes/ratnayn/Data/instanseg/slurm_runs/'
    'instanseg_multihead_0325_20260825/source/instanseg',
)).expanduser().resolve()
PUBLIC_MODEL_CACHE_ROOT = Path(os.environ.get(
    'INSTANSEG_PUBLIC_MODEL_CACHE',
    str(TRAINING_ROOT / 'public_model_cache'),
)).expanduser().resolve()
PUBLIC_MODEL_VERSION = '0.1.1'
RESULTS_ROOT = Path(os.environ.get(
    'INSTANSEG_SLIDE0330_GALLERY_ROOT',
    '/data1/lowes/ratnayn/Codex/codex-scratch/mIF-pipeline/'
    'slide0330_model_gallery',
)).expanduser().resolve()

DEVICE = os.environ.get('INSTANSEG_VISUALIZATION_DEVICE', 'cuda:0')
if not torch.cuda.is_available():
    raise RuntimeError('Select a CUDA notebook kernel before running model inference.')

# User controls. Crops are selected once with this seed and then reused for every model.
CROP_SIZE_PX = 768
N_CROPS = 10
RANDOM_SEED = 3300330
RESOLVE_CELL_AND_NUCLEUS = False
SAVE_FIGURES = True

# Shared visualization postprocessing. Change only if a deliberate visual ablation is wanted.
POSTPROCESSING = {
    'min_size': 10,
    'mask_threshold': 0.3,
    'peak_distance': 5,
    'seed_threshold': 0.5,
    'overlap_threshold': 0.3,
    'mean_threshold': 0.0,
    'fg_threshold': 0.5,
    'window_size': 32,
    'cleanup_fragments': False,
    'max_seeds': 2000,
    'resolve_cell_and_nucleus': RESOLVE_CELL_AND_NUCLEUS,
}

MODEL_SPECS = [
    {
        'name': 'cpdmi_050_t256_w128_heavy_multihead',
        'label': 'CPDMI-050', 'kind': 'trained', 'pixel_size_um': 0.5,
    },
    {
        'name': 'cpdmi_0325_t256_w128_heavy_multihead',
        'label': 'CPDMI-0325 t256', 'kind': 'trained', 'pixel_size_um': 0.325,
    },
    {
        'name': 'cpdmi_0325_t384_w128_heavy_multihead',
        'label': 'CPDMI-0325 t384', 'kind': 'trained', 'pixel_size_um': 0.325,
    },
    {
        'name': 'cpdmi_tissuenet_0325_t256_w128_heavy_drop_multihead',
        'label': 'Mixed CPDMI+TissueNet', 'kind': 'trained', 'pixel_size_um': 0.325,
    },
    {
        'name': 'fluorescence_nuclei_and_cells',
        'label': 'Public fluorescence', 'kind': 'public', 'pixel_size_um': 0.5,
    },
]
MODEL_LABELS = [spec['label'] for spec in MODEL_SPECS]
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print({'slide': FULL_MERGE_OME, 'device': DEVICE, 'models': MODEL_LABELS,
       'resolve_cell_and_nucleus': RESOLVE_CELL_AND_NUCLEUS})

In [ ]:
if not FULL_MERGE_OME.is_file():
    raise FileNotFoundError(FULL_MERGE_OME)
if not (SOURCE_ROOT / 'instanseg').is_dir():
    raise FileNotFoundError(SOURCE_ROOT / 'instanseg')
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

import instanseg
from instanseg import InstanSeg as NativeInstanSeg
from tiffslide import TiffSlide
import instanseg.inference_class as instanseg_inference_class

# Preserve the validated TiffSlide substitution used by the watershed test.
instanseg_inference_class.TiffSlide = TiffSlide

instanseg_import_path = Path(instanseg.__file__).resolve()
if not instanseg_import_path.is_relative_to(SOURCE_ROOT / 'instanseg'):
    raise RuntimeError(f'Imported InstanSeg from {instanseg_import_path}, not {SOURCE_ROOT}.')
print({'instanseg_source': str(instanseg_import_path), 'torch': torch.__version__,
       'gpu': torch.cuda.get_device_name(torch.cuda.current_device())})

## Load the SLIDE-0330 channels and choose identical crops

The source is opened through `tifffile`/Zarr so only the requested crop tiles are read. The crop coordinates are selected deterministically from random candidates with a DAPI signal screen to avoid showing mostly blank regions.

In [ ]:
SEGMENTATION_CHANNELS = [
    'R1_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488',
    'R6_CD45_CST_AF647', 'R6_PANCK_AE1_AE3_750',
    'R12_CD31_D8V9E_AF750', 'R7_NAK_ATPASE_555',
    'R8_PODOPLANIN_750', 'R8_F480_D2S9R_555',
    'R9_CD68_E3O7V_488',
    'R12_CD3E_E4T1B_AF555',
]
REFERENCE_PIXEL_SIZE_UM = 0.325

def channel_names_from_ome_xml(xml):
    return re.findall(r'<Channel\b[^>]*Name="([^"]+)"', xml or '')

with tifffile.TiffFile(FULL_MERGE_OME) as tf:
    source_names = channel_names_from_ome_xml(tf.ome_metadata)
    level0 = tf.series[0].levels[0]
    source_shape = tuple(int(v) for v in level0.shape)
    source_channel_to_index = {name: i for i, name in enumerate(source_names)}

missing = [name for name in SEGMENTATION_CHANNELS if name not in source_channel_to_index]
if missing:
    raise ValueError(f'Missing SLIDE-0330 channels: {missing}')
CHANNEL_IDS = [source_channel_to_index[name] for name in SEGMENTATION_CHANNELS]
DAPI_INDEX = SEGMENTATION_CHANNELS.index('R1_DAPI')
DISPLAY_INDEX = {
    'red': SEGMENTATION_CHANNELS.index('R6_CD45_CST_AF647'),
    'green': SEGMENTATION_CHANNELS.index('R7_NAK_ATPASE_555'),
    'blue': SEGMENTATION_CHANNELS.index('R1_DAPI'),
}
print({'source_shape': source_shape, 'source_channel_count': len(source_names),
       'selected_channel_ids': CHANNEL_IDS, 'selected_channels': SEGMENTATION_CHANNELS})

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
source_height, source_width = source_shape[-2:]
if CROP_SIZE_PX >= source_height or CROP_SIZE_PX >= source_width:
    raise ValueError(f'Crop size {CROP_SIZE_PX} does not fit source shape {source_shape[-2:]}')

candidate_count = max(24, N_CROPS * 10)
candidate_coords = []
candidate_scores = []
crops_by_candidate = {}

with tifffile.TiffFile(FULL_MERGE_OME) as tf:
    store = tf.series[0].aszarr(level=0)
    try:
        source = zarr.open(store, mode='r')
        for _ in range(candidate_count):
            x = int(rng.integers(0, source_width - CROP_SIZE_PX))
            y = int(rng.integers(0, source_height - CROP_SIZE_PX))
            dapi = np.asarray(source[CHANNEL_IDS[DAPI_INDEX], y:y + CROP_SIZE_PX, x:x + CROP_SIZE_PX])
            score = float(np.percentile(dapi, 99.5))
            candidate_coords.append((x, y))
            candidate_scores.append(score)
            crops_by_candidate[(x, y)] = dapi

        scores = np.asarray(candidate_scores)
        eligible = np.flatnonzero(scores >= np.quantile(scores, 0.5))
        chosen_indices = rng.choice(eligible, size=min(N_CROPS, len(eligible)), replace=False)
        if len(chosen_indices) < N_CROPS:
            raise RuntimeError('Not enough candidate crops were generated.')

        CROP_SPECS = []
        CROP_IMAGES = []
        for crop_number, candidate_index in enumerate(chosen_indices, start=1):
            x, y = candidate_coords[int(candidate_index)]
            crop = np.stack([
                np.asarray(source[channel_id, y:y + CROP_SIZE_PX, x:x + CROP_SIZE_PX])
                for channel_id in CHANNEL_IDS
            ]).astype(np.float32, copy=False)
            CROP_SPECS.append({
                'crop': crop_number, 'x': x, 'y': y,
                'width': CROP_SIZE_PX, 'height': CROP_SIZE_PX,
                'dapi_99_5': float(candidate_scores[int(candidate_index)]),
            })
            CROP_IMAGES.append(crop)
    finally:
        store.close()

CROP_IMAGES = [np.ascontiguousarray(crop) for crop in CROP_IMAGES]
display(pd.DataFrame(CROP_SPECS))
print('Each model will receive these exact crop coordinates and channel order.')

## Export self-trained checkpoints to native TorchScript

The existing InstanSeg export path converts each raw checkpoint into `InstanSeg_Torchscript`, including the native embedding postprocessing and optional cell/nucleus reconciliation. The resulting `.pt` file can be passed directly to `InstanSeg`; no notebook-specific model wrapper is needed.

In [ ]:
EXPORT_ROOT = RESULTS_ROOT / 'torchscripts'
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

# These are the paths consumed by export_to_torchscript(). The official
# export helper currently reads the example image to establish the script
# signature; it falls back to a correctly sized random input when channel
# count differs from the example.
os.environ['INSTANSEG_MODEL_PATH'] = str(MODEL_ROOT)
os.environ['INSTANSEG_TORCHSCRIPT_PATH'] = str(EXPORT_ROOT)
os.environ['EXAMPLE_IMAGE_PATH'] = str(SOURCE_ROOT / 'instanseg' / 'examples')
from instanseg.utils.utils import export_to_torchscript

def export_and_load_trained_runner(spec):
    export_path = EXPORT_ROOT / f"{spec['name']}.pt"
    if not export_path.is_file():
        print(f"Exporting {spec['label']} ...", flush=True)
        export_to_torchscript(
            spec['name'],
            show_example=False,
            torchscript_name=spec['name'],
            use_optimized_params=False,
        )
    if not export_path.is_file():
        raise FileNotFoundError(export_path)
    script = torch.jit.load(str(export_path))
    exported_pixel_size = float(script.pixel_size)
    expected_pixel_size = float(spec['pixel_size_um'])
    if abs(exported_pixel_size - expected_pixel_size) > 1e-6:
        raise ValueError(
            f"{spec['name']} exports at {exported_pixel_size} µm, "
            f"expected {expected_pixel_size} µm."
        )
    if not bool(script.cells_and_nuclei):
        raise ValueError(f"{spec['name']} is not a dual-head cells-and-nuclei model.")
    return NativeInstanSeg(
        model_type=script, device=DEVICE, verbosity=0, channels_last=False
    )

print('Self-trained checkpoints will use the official InstanSeg TorchScript export path.')

In [ ]:
def load_runner(spec):
    if spec['kind'] == 'public':
        model_index_path = SOURCE_ROOT / 'instanseg' / 'bioimageio_models' / 'model-index.json'
        entries = [entry for entry in json.loads(model_index_path.read_text()) if entry.get('name') == spec['name']]
        if not entries or entries[0].get('version') != PUBLIC_MODEL_VERSION:
            raise RuntimeError(f'Expected public {spec["name"]} version {PUBLIC_MODEL_VERSION}.')
        PUBLIC_MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
        os.environ['INSTANSEG_BIOIMAGEIO_PATH'] = str(PUBLIC_MODEL_CACHE_ROOT)
        return NativeInstanSeg(
            model_type=spec['name'], device=DEVICE, verbosity=0, channels_last=False
        )
    return export_and_load_trained_runner(spec)

def predict_crop(runner, crop):
    image = torch.from_numpy(crop)
    with torch.inference_mode():
        labels = runner.eval_small_image(
            image, pixel_size=REFERENCE_PIXEL_SIZE_UM, normalise=True,
            return_image_tensor=False, target='all_outputs',
            rescale_output=True, **POSTPROCESSING,
        )
    labels = labels.squeeze(0).to(torch.int32).cpu().numpy()
    if labels.shape[0] != 2 or labels.shape[-2:] != crop.shape[-2:]:
        raise ValueError(f'Unexpected prediction shape {labels.shape} for crop {crop.shape}.')
    return labels

PREDICTIONS = {crop_number: {} for crop_number in range(1, len(CROP_IMAGES) + 1)}
MODEL_METADATA = []
started = time.perf_counter()
for spec in MODEL_SPECS:
    print(f'[{spec["label"]}] loading', flush=True)
    runner = load_runner(spec)
    try:
        for crop_number, crop in enumerate(CROP_IMAGES, start=1):
            PREDICTIONS[crop_number][spec['label']] = predict_crop(runner, crop)
            print(f'  crop {crop_number}/{len(CROP_IMAGES)} complete', flush=True)
        MODEL_METADATA.append({
            'label': spec['label'], 'kind': spec['kind'],
            'pixel_size_um': spec['pixel_size_um'],
            'native_api': 'eval_small_image',
            'resolve_cell_and_nucleus': RESOLVE_CELL_AND_NUCLEUS,
        })
    finally:
        del runner
        gc.collect()
        torch.cuda.empty_cache()
print(f'Completed in {(time.perf_counter() - started) / 60:.1f} minutes.')

provenance = {
    'slide_id': SLIDE_ID, 'source_image': str(FULL_MERGE_OME),
    'source_pixel_size_um': REFERENCE_PIXEL_SIZE_UM,
    'channels': SEGMENTATION_CHANNELS, 'channel_ids': CHANNEL_IDS,
    'crop_specs': CROP_SPECS, 'random_seed': RANDOM_SEED,
    'postprocessing': POSTPROCESSING, 'models': MODEL_METADATA,
}
(RESULTS_ROOT / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
print('Saved provenance:', RESULTS_ROOT / 'provenance.json')

## Visualize identical crops across models

Cyan boundaries are nuclei and yellow boundaries are whole cells. The second and third rows isolate the nuclear and cell heads so differences in one compartment are easier to see.

In [ ]:
def robust01(image, percentiles=(1.0, 99.8)):
    image = np.asarray(image, dtype=np.float32)
    low, high = np.percentile(image, percentiles)
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)
    return np.clip((image - low) / (high - low), 0, 1)

def display_rgb(crop):
    return np.stack([
        robust01(crop[DISPLAY_INDEX['red']]),
        robust01(crop[DISPLAY_INDEX['green']]),
        robust01(crop[DISPLAY_INDEX['blue']]),
    ], axis=-1)

def show_prediction(ax, rgb, labels=None, mode='both', title=None):
    ax.imshow(rgb)
    if labels is not None and mode in ('both', 'nuclei'):
        nuclei_boundary = find_boundaries(labels[0], mode='outer')
        ax.contour(nuclei_boundary, levels=[0.5], colors=['cyan'], linewidths=0.45)
    if labels is not None and mode in ('both', 'cells'):
        cell_boundary = find_boundaries(labels[1], mode='outer')
        ax.contour(cell_boundary, levels=[0.5], colors=['yellow'], linewidths=0.45)
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=9)

def plot_crop_gallery(crop_number):
    crop = CROP_IMAGES[crop_number - 1]
    rgb = display_rgb(crop)
    columns = ['Input'] + MODEL_LABELS
    figure, axes = plt.subplots(3, len(columns), figsize=(6.1 * len(columns), 19), squeeze=False)
    show_prediction(axes[0, 0], rgb, title=f'Input\n{x_label(crop_number)}')
    axes[1, 0].imshow(rgb); axes[1, 0].set_axis_off(); axes[1, 0].set_title('Input', fontsize=9)
    axes[2, 0].imshow(rgb); axes[2, 0].set_axis_off(); axes[2, 0].set_title('Input', fontsize=9)
    for column, label in enumerate(MODEL_LABELS, start=1):
        labels = PREDICTIONS[crop_number][label]
        show_prediction(axes[0, column], rgb, labels, mode='both', title=label)
        show_prediction(axes[1, column], rgb, labels, mode='nuclei', title='Nuclei')
        show_prediction(axes[2, column], rgb, labels, mode='cells', title='Cells')
    axes[0, 0].set_ylabel('Both', fontsize=10)
    axes[1, 0].set_ylabel('Nuclei', fontsize=10)
    axes[2, 0].set_ylabel('Cells', fontsize=10)
    figure.suptitle(
        f'{SLIDE_ID} crop {crop_number} | resolve={RESOLVE_CELL_AND_NUCLEUS} | '
        'cyan=nuclei, yellow=cells', fontsize=13
    )
    figure.tight_layout()
    if SAVE_FIGURES:
        output_path = RESULTS_ROOT / f'{SLIDE_ID}_crop{crop_number}_gallery.png'
        figure.savefig(output_path, dpi=160, bbox_inches='tight')
        print('Saved:', output_path)
    plt.show()

def x_label(crop_number):
    spec = CROP_SPECS[crop_number - 1]
    return f"x={spec['x']}, y={spec['y']}"

for crop_number in range(1, len(CROP_IMAGES) + 1):
    plot_crop_gallery(crop_number)